# Pretty Formula One

This notebook is to fetch the last race to update the database on the website

In [ ]:
YEAR = 2026

In [ ]:
import fastf1
import json
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
    
output_dir = f'./data/{YEAR}'
os.makedirs(output_dir, exist_ok=True)
filename = f"{output_dir}/rounds_{YEAR}.json"

with open(f"{output_dir}/drivers_{YEAR}.json", "r") as f:
    drivers_list = json.load(f)

schedule_df = fastf1.get_event_schedule(YEAR)
races = schedule_df["RoundNumber"][schedule_df["RoundNumber"] > 0].tolist()

for ROUND_ID in races:
    existing_rounds = []
    if os.path.exists(filename):
        with open(filename, "r") as f:
            try:
                existing_rounds = json.load(f)
            except json.JSONDecodeError:
                existing_rounds = []

    if any(r['id'] == int(ROUND_ID) for r in existing_rounds):
        print(f"Skipping round {ROUND_ID}: Already processed.")
        continue

    try:
        race_session = fastf1.get_session(YEAR, ROUND_ID, "R")
        race_session.load(telemetry=False, weather=False)
        
        race_points_map = {
            f"{row['DriverId']}_{YEAR}": int(row['Points']) 
            for _, row in race_session.results.iterrows()
        }

        id_to_number_map = {
            f"{row['DriverId']}_{YEAR}": row['DriverNumber'] 
            for _, row in race_session.results.iterrows()
        }

        sprint_points_map = {}
        has_sprint = False
        try:
            sprint_session = fastf1.get_session(YEAR, ROUND_ID, "S")
            sprint_session.load(laps=False, telemetry=False, weather=False)
            has_sprint = True
            sprint_points_map = {
                row['DriverNumber']: int(row['Points']) 
                for _, row in sprint_session.results.iterrows()
            }
        except Exception:
            pass

        race_results_list = []
        for driver in drivers_list:
            d_id = driver["id"]
            r_points = race_points_map.get(d_id, 0)
            s_points = 0
            d_number = id_to_number_map.get(d_id)
            
            if has_sprint and d_number:
                s_points = sprint_points_map.get(d_number, 0)
                    
            laps = race_session.laps
            tyre_data = laps[['Compound', 'LapNumber', 'DriverNumber', 'Stint']]
            lap_tyre_data = {}
            
            for _, row in tyre_data.iterrows():
                if str(row["Compound"]) == "nan":
                    continue
                driver_num = row["DriverNumber"]
                if driver_num not in lap_tyre_data:
                    lap_tyre_data[driver_num] = []
                
                last_entry = lap_tyre_data[driver_num][-1] if lap_tyre_data[driver_num] else None
                if last_entry and row["Compound"] == last_entry["compound"]:
                    last_entry["lapEnd"] = int(row["LapNumber"])
                else:
                    lap_tyre_data[driver_num].append({
                        "compound": row["Compound"],
                        "lapStart": int(row["LapNumber"]),
                        "lapEnd": int(row["LapNumber"]),
                        "stint": int(row["Stint"])
                    })

            race_results_list.append({
                "driver_id": d_id,
                "racePoints": r_points,
                "sprintPoints": s_points,
                "tyre_strat": lap_tyre_data.get(d_number, [])
            })

        event_info = race_session.event
        country = event_info["Country"]
        total_rounds = len(races)

        round_data = {
            "id": int(event_info["RoundNumber"]),
            "year": YEAR,
            "index": int(event_info["RoundNumber"]),
            "totalRounds": total_rounds,
            "name": event_info["EventName"],
            "nameVerbose": event_info["OfficialEventName"],
            "country": country,
            "backgroundImage": f"/assets/circuits/{country.lower().replace(' ', '_')}.png",
            "results": sorted(race_results_list, key=lambda x: x["racePoints"], reverse=True),
        }

        existing_rounds.append(round_data)
        existing_rounds.sort(key=lambda x: x["id"])
        
        with open(filename, "w") as f:
            json.dump(existing_rounds, f, separators=(',', ':'))
            
        print(f"Successfully processed and saved round {ROUND_ID}: {event_info['EventName']}")

    except Exception as e:
        print(f"Error processing round {ROUND_ID}: {e}")
        break

In [13]:
from fastf1.ergast import Ergast
import fastf1
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

ergast = Ergast()
DRIVERS = ergast.get_driver_info(YEAR)["driverId"].to_list()

schedule = fastf1.get_event_schedule(YEAR)
races = schedule["RoundNumber"][schedule["RoundNumber"] > 0].tolist()

already_processed = {i: 0 for i in races}
telemetries = [f for f in os.listdir(f'data/{YEAR}') if f.startswith('telemetry_')]
for file in telemetries:
    race_id_str = file.split(f'_{YEAR}_')[1].split('.')[0]
    race_id = int(race_id_str)
    already_processed[race_id] += 1

last_race = 0
for k, v in already_processed.items():
    if v == len(DRIVERS):
        last_race = k
next_race = last_race + 1 if last_race + 1 < len(races) else None

if not next_race:
    raise Exception("All races have been processed.")

unprocessed_races = [race for race in races if race >= next_race]

for race in unprocessed_races:
    for driver in DRIVERS:
        session = fastf1.get_session(YEAR, race, "R")
        session.load(laps=True, telemetry=True, weather=False)
        driver_slug = f"{driver}_{YEAR}"
        for _, row in session.results.iterrows():
            if row["DriverId"] == driver:
                driver_abv = row["Abbreviation"]
                break
        driver_laps = session.laps.pick_drivers(driver_abv)
        if driver_laps.empty:
            continue
        fastest_lap = driver_laps.pick_fastest()
        if fastest_lap is None:
            fastest_lap = driver_laps.iloc[0]
        start_td = fastest_lap["Time"] - fastest_lap["LapTime"]
        end_td = fastest_lap["Time"]
        try:
            telemetry = fastest_lap.get_telemetry()
        except Exception:
            raise Exception(f"No telemetry data for driver {driver} in race {race}.")
        telemetry['RelativeSeconds'] = (telemetry['SessionTime'] - start_td).dt.total_seconds()
        telemetry['Compound'] = fastest_lap['Compound']

        position_points = []
        for row in telemetry.itertuples():
            position_points.append((
                round(row.RelativeSeconds,1), 
                round(row.X,1), 
                round(row.Y,1),
                round(row.Z,1),
                round(row.RPM,1),
                round(row.Speed,1), 
                row.nGear, 
                round(row.Throttle,1), 
                row.Brake, 
                row.DRS, 
                row.Status,
                row.Compound
            ))

        os.makedirs(f'data/{YEAR}', exist_ok=True)
        filename = f'data/{YEAR}/telemetry_{driver}_{YEAR}_{race}.csv'

        with open(filename, 'w') as f:
            f.write("seconds,x,y,z,rpm,speed,gear,throttle,brake,drs,status,compound\n")
            for point in position_points:
                f.write(','.join(map(str, point)) + '\n')
                
        d_nam = driver
        d_idx = DRIVERS.index(driver)+1
        d_tot = len(DRIVERS)
        r_idx = races.index(race)+1
        r_tot = len(races)
        t_idx = d_idx*r_idx
        t_tot = d_tot*r_tot
        print(f"Exported points for {d_nam} ({d_idx}/{d_tot}) in race {r_idx} ({r_idx}/{r_tot}) total iterations: [{t_idx}/{t_tot}]")

Exception: No telemetry data for driver albon in race 6.